# От простого запроса к автономной системе: пишем ИИ‑исследователя с нуля

Автор: [Сергей Тращенков](https://t.me/gigatrash), команда [GigaChain](https://github.com/ai-forever/gigachain), Сбер.

Вдохновлено проектом Ivan Leo ["Building a Deep Research Agent from Scratch"](https://github.com/ivanleomk/deep-research-interactions)

Исследовательские задачи плохо укладываются в схему «вопрос → ответ». Чтобы подготовить содержательный результат, системе приходится сначала понять задачу, затем составить план, найти информацию в нескольких источниках, проверить её, собрать промежуточные выводы и только после этого сформировать итоговый отчёт.

Именно так работают современные исследовательские агенты. Вместо одного обращения к модели они выполняют цепочку связанных действий: планируют работу, используют инструменты поиска, находят информацию, отбирают только релевантные данные, принимают промежуточные решения и постепенно продвигаются к результату.

На мастер-классе мы соберём небольшую учебную версию такой системы. Агент будет принимать вопрос, строить план, искать факты через Tavily, делегировать часть поиска подагенту, собирать итоговый отчёт и оставлять понятные трейсы в Arize Phoenix.

> **Агент = Модель + Обвязка (harness).**

Под harness обычно понимают весь программный слой вокруг модели: цикл выполнения агента, управление состоянием и памятью, работу с инструментами, планирование и маршрутизацию задач, взаимодействие между агентами, обработку ошибок, ограничения по времени и стоимости, механизмы безопасности, а также наблюдаемость и логирование.

На практике полноценный harness может быть очень большим. На мастер-классе мы реализуем только часть этих компонентов. Мы сосредоточимся на базовой механике агентной системы и шаг за шагом соберём её самостоятельно на Python. Это позволит увидеть, как взаимодействуют модель, инструменты и состояние задачи, и понять, из каких компонентов складывается работа исследовательского агента.

## 1. Настройка: зависимости и безопасный ввод ключей

Нам нужны:

- **[OpenRouter](https://openrouter.ai/)** — единая точка доступа к разным моделям, совместимая с OpenAI API. Мы будем использовать OpenAI SDK, но направим его на `https://openrouter.ai/api/v1`.
- **[Tavily](https://app.tavily.com/)** — поисковый API для ИИ-приложений. Он возвращает результаты в удобном для агента виде: заголовок, ссылку, фрагмент, релевантность и, при необходимости, содержимое страницы.
- **[Phoenix](https://arize.com/docs/phoenix) / OpenInference** — локальная наблюдаемость: трейс показывает весь запуск, спан — отдельный шаг внутри трейса.

Исследовательскому агенту обязательно нужен ключ Tavily: без внешнего поиска он не сможет собирать актуальные источники. Зарегистрируйтесь в Tavily, откройте раздел API keys, создайте или скопируйте ключ и добавьте его в `.env`.

### Настройка через `.env`

Чтобы не вводить ключи вручную при каждом запуске, скопируйте шаблон:

```bash
cp .env.example .env
# откройте .env и вставьте OPENROUTER_API_KEY / TAVILY_API_KEY
```

Блокнот автоматически подхватит `.env`, если он лежит в корне проекта или в текущей папке запуска. Настоящий `.env` не коммитьте: в нём хранятся секреты.

Локальная воспроизводимая установка из корня этой папки. Целевая версия — Python 3.12+:

```bash
brew install python@3.12  # если python3.12 ещё нет
/opt/homebrew/bin/python3.12 -m venv .venv
. .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
phoenix serve  # в отдельном терминале, если хотите видеть трейсы в UI
```


<!-- GAP-маркеры для офлайн-проверки: GAP 1 — tool registry wiring; GAP 2 — todo/state update; GAP 3 — delegate_search/subagent wiring. -->


### Модели OpenRouter: платные, бесплатные и лимиты

По умолчанию в этом блокноте используется модель из `.env`:

```bash
OPENROUTER_MODEL=z-ai/glm-5.1
OPENROUTER_FALLBACK_MODEL=openrouter/auto
```

Для экспериментов без списания денег можно выбрать бесплатную OpenRouter-модель: у таких моделей ID обычно заканчивается на `:free`.
Например:

```bash
OPENROUTER_MODEL=openai/gpt-oss-120b:free
OPENROUTER_FALLBACK_MODEL=openai/gpt-oss-120b:free
```

Если вы хотите **строго бесплатный запуск**, задавайте `:free` и для основной модели, и для fallback. Иначе `openrouter/auto` может выбрать не бесплатный маршрут.

Что важно знать про лимиты OpenRouter free-моделей:

- free-варианты доступны без стоимости, но могут иметь более низкую доступность и отдельные rate limits;
- актуальные лимиты OpenRouter для `:free` моделей: до **20 запросов в минуту**;
- если на аккаунте куплено меньше $10 credits — обычно **50 free-model requests/day**;
- после покупки хотя бы $10 credits лимит free-моделей повышается до **1000 requests/day**;
- популярные free-модели могут временно отвечать `429` из-за upstream rate limit даже если ваш ключ ещё не исчерпал дневной лимит.

Ссылки: [OpenRouter Free Variant](https://openrouter.ai/docs/guides/routing/model-variants/free), [OpenRouter API Rate Limits](https://openrouter.ai/docs/api-reference/limits/), [OpenRouter Pricing / FAQ](https://openrouter.ai/pricing).

> ⚠️ **Предупреждение для мастер-класса.** Бесплатные модели полезны для обучения и проверки механики агента, но качество может быть заметно хуже, чем у основной модели: слабее tool calling, хуже следование инструкциям, больше риск галлюцинаций, сломанных ссылок и неполного отчёта. Один полный запуск нашего deep-research агента может потратить значимую часть дневного лимита 50 запросов, поэтому на полностью бесплатном аккаунте обычно хватает на один аккуратный прогон, а не на серию улучшений и повторных запусков.


In [ ]:
# Если вы запускаете notebook в чистом окружении, установите зависимости.
# Из корня проекта:
# %pip install -q -r requirements.txt
# Если notebook открыт из папки notebooks/:
# %pip install -q -r ../requirements.txt

import json
import os
import re
import textwrap
from dataclasses import dataclass, field
from datetime import date
from getpass import getpass
from pathlib import Path
from typing import Any, Callable, Literal, Optional

try:
    from openai import OpenAI
except Exception:  # пакет может быть не установлен до install cell
    OpenAI = None


def parse_dotenv_value(raw: str) -> str:
    """Минимальный parser .env без дополнительной зависимости python-dotenv."""
    value = raw.strip()
    if not value:
        return ""
    if value[0] in {"'", '"'} and value[-1:] == value[0]:
        return value[1:-1]
    # Для незакрытых кавычками значений поддерживаем inline comments: KEY=value # comment
    return value.split(" #", 1)[0].strip()


def load_dotenv_file(path: Optional[Path] = None, *, override: bool = False) -> Optional[Path]:
    """Загрузить .env из корня проекта/текущей папки, не перезаписывая уже заданные env vars."""
    if path is None:
        cwd = Path.cwd()
        candidates = [cwd / ".env", cwd.parent / ".env"]
        path = next((candidate for candidate in candidates if candidate.exists()), None)
    if path is None or not path.exists():
        return None

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        if line.startswith("export "):
            line = line[len("export "):].strip()
        key, _, raw_value = line.partition("=")
        key = key.strip()
        if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", key):
            continue
        if override or key not in os.environ:
            os.environ[key] = parse_dotenv_value(raw_value)
    return path


DOTENV_PATH = None if os.getenv("DISABLE_DOTENV") == "1" else load_dotenv_file()
if os.getenv("DISABLE_DOTENV") == "1":
    print("Загрузка .env отключена через DISABLE_DOTENV=1.")
elif DOTENV_PATH:
    print(f"Загружен .env: {DOTENV_PATH}")
else:
    print(".env не найден — используем только переменные окружения.")

TODAY = date.today().isoformat()
PRIMARY_MODEL = os.getenv("OPENROUTER_MODEL", "z-ai/glm-5.1")
FALLBACK_MODEL = os.getenv("OPENROUTER_FALLBACK_MODEL", "openrouter/auto")
PROJECT_NAME = "codefest-ai-researcher"

# Для мастер-класса ключи лучше вводить через getpass или переменные окружения,
# а не сохранять в notebook.
if not os.getenv("OPENROUTER_API_KEY"):
    print("OPENROUTER_API_KEY не найден. Финальный агентный запуск потребует live model API.")
    # os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

if not os.getenv("TAVILY_API_KEY"):
    print("TAVILY_API_KEY не найден. Финальный агентный запуск потребует настоящий Tavily search.")
    # os.environ["TAVILY_API_KEY"] = getpass("Tavily API key: ")

LIVE_MODEL = bool(os.getenv("OPENROUTER_API_KEY") and OpenAI is not None)
LIVE_SEARCH = bool(os.getenv("TAVILY_API_KEY"))
print({"LIVE_MODEL": LIVE_MODEL, "LIVE_SEARCH": LIVE_SEARCH, "TODAY": TODAY})



## 2. Наблюдаемость (observability): видим работу агента по шагам

Агентная система недетерминирована: один и тот же вопрос может привести к разным промежуточным шагам, вызовам инструментов и даже разному итоговому ответу. Поэтому при разработке агента важно смотреть не только на финальный результат, но и на весь путь выполнения.

Внутри запуска происходит много технически важных вещей: модель получает контекст, принимает решения, вызывает инструменты, передаёт аргументы, получает результаты поиска, тратит токены, обновляет состояние и постепенно собирает ответ. Обычных `print`-логов быстро становится недостаточно: информации много, она вложенная, её нужно сохранять, пересматривать, сравнивать между запусками и использовать для отладки.

Для этого используют системы трейсинга. В этом блокноте мы подключим **Arize Phoenix** — локальный инструмент наблюдаемости для LLM-приложений.

В Phoenix один запуск агента отображается как **trace** — трейс. Например: «подготовить исследовательский ответ на вопрос пользователя». Отдельные шаги внутри запуска отображаются как **spans** — спаны: вызов модели, обращение к Tavily, запуск подагента, обработка результата инструмента или сборка итогового отчёта.

Phoenix даёт UI для просмотра запусков, хранит данные для последующего анализа и помогает понять, где именно система повела себя не так: модель выбрала неправильный инструмент, инструмент вернул пустой результат, контекст стал слишком большим или итоговый ответ собрался из неудачных промежуточных данных.

Возможности Phoenix этим не ограничиваются: такие системы также могут помогать с датасетами, оценкой качества, экспериментами и управлением промптами. Но в рамках мастер-класса мы используем главное: трейсы, чтобы видеть работу агента шаг за шагом.


Phoenix также помогает объяснить две практические проблемы длинных агентных запусков: **context rot** (порча контекста из-за лишних/устаревших данных) и **context anxiety** (непонимание, что именно сейчас находится в контексте агента).


In [ ]:
tracer = None
tracer_provider = None
PHOENIX_UI_URL = None
PHOENIX_ENDPOINT = None
PHOENIX_SESSION = None
PHOENIX_FORCE_FLUSH = os.getenv("PHOENIX_FORCE_FLUSH", "1") != "0"

try:
    from opentelemetry.trace import Status, StatusCode
except Exception:
    Status = None
    StatusCode = None


def is_port_open(host: str, port: int, timeout: float = 0.5) -> bool:
    import socket

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0


def check_url(url: str, timeout: float = 2.0) -> Optional[int]:
    from urllib.request import urlopen

    try:
        with urlopen(url, timeout=timeout) as response:
            return int(response.status)
    except Exception:
        return None


def normalize_phoenix_endpoint(raw_endpoint: Optional[str], port: int) -> str:
    endpoint = (raw_endpoint or f"http://127.0.0.1:{port}/v1/traces").strip().rstrip("/")
    if not endpoint.endswith("/v1/traces"):
        endpoint += "/v1/traces"
    # 127.0.0.1 avoids occasional localhost IPv6/IPv4 mismatch on macOS.
    endpoint = endpoint.replace("http://localhost:", "http://127.0.0.1:")
    return endpoint


def is_local_phoenix_endpoint(endpoint: str) -> bool:
    from urllib.parse import urlparse

    host = urlparse(endpoint).hostname
    return host in {"localhost", "127.0.0.1", "0.0.0.0", None}


try:
    if os.getenv("DISABLE_PHOENIX") == "1":
        raise RuntimeError("Phoenix disabled by DISABLE_PHOENIX=1")

    import phoenix as px
    from phoenix.otel import register

    phoenix_port = int(os.getenv("PHOENIX_PORT", "6006"))
    phoenix_grpc_port = int(os.getenv("PHOENIX_GRPC_PORT", "4317"))
    phoenix_working_dir = os.getenv("PHOENIX_WORKING_DIR", str(Path.cwd() / ".phoenix"))
    PHOENIX_ENDPOINT = normalize_phoenix_endpoint(os.getenv("PHOENIX_COLLECTOR_ENDPOINT"), phoenix_port)
    PHOENIX_UI_URL = f"http://127.0.0.1:{phoenix_port}/"

    if is_local_phoenix_endpoint(PHOENIX_ENDPOINT):
        if is_port_open("127.0.0.1", phoenix_port):
            status = check_url(PHOENIX_UI_URL)
            print(f"Используем уже запущенный Phoenix: {PHOENIX_UI_URL} HTTP {status or 'не ответил'}")
        else:
            if is_port_open("127.0.0.1", phoenix_grpc_port):
                raise RuntimeError(
                    f"gRPC port {phoenix_grpc_port} занят, а HTTP port {phoenix_port} свободен. "
                    "Перезапустите kernel или поменяйте PHOENIX_PORT/PHOENIX_GRPC_PORT в .env."
                )
            os.environ["PHOENIX_PORT"] = str(phoenix_port)
            os.environ["PHOENIX_GRPC_PORT"] = str(phoenix_grpc_port)
            os.environ["PHOENIX_WORKING_DIR"] = phoenix_working_dir
            Path(phoenix_working_dir).mkdir(parents=True, exist_ok=True)
            PHOENIX_SESSION = px.launch_app(use_temp_dir=False)
            if PHOENIX_SESSION is None:
                raise RuntimeError("phoenix.launch_app() не вернул сессию")
            PHOENIX_UI_URL = getattr(PHOENIX_SESSION, "url", PHOENIX_UI_URL)
            PHOENIX_ENDPOINT = f"http://127.0.0.1:{phoenix_port}/v1/traces"
            status = check_url(PHOENIX_UI_URL)
            print(f"Локальный Phoenix запущен: {PHOENIX_UI_URL} HTTP {status or 'стартует'}")
            print(f"Phoenix DB dir: {phoenix_working_dir}")
    else:
        print(f"Используем внешний Phoenix collector: {PHOENIX_ENDPOINT}")

    tracer_provider = register(
        project_name=PROJECT_NAME,
        endpoint=PHOENIX_ENDPOINT,
        protocol="http/protobuf",
        auto_instrument=True,
        batch=False,
    )
    tracer = tracer_provider.get_tracer(__name__)
    print(f"Phoenix tracing enabled. UI: {PHOENIX_UI_URL or 'external'}, OTLP HTTP endpoint: {PHOENIX_ENDPOINT}")
    print(f"Phoenix force flush after spans: {PHOENIX_FORCE_FLUSH}")
except Exception as exc:
    print("Phoenix не включён — продолжаем без live traces.")
    print("Чтобы включить: установите arize-phoenix/arize-phoenix-otel; notebook умеет сам поднять локальный Phoenix.")
    print(type(exc).__name__, str(exc)[:300])

def _span_preview(value: Any, limit: int = 8000) -> str:
    if isinstance(value, str):
        text = value
    else:
        try:
            text = json.dumps(value, ensure_ascii=False, default=str)
        except Exception:
            text = str(value)
    text = text.strip()
    if len(text) <= limit:
        return text
    return text[:limit].rstrip() + "…"


def _span_mime_type(value: Any) -> str:
    return "text/plain" if isinstance(value, str) else "application/json"


def _span_is_recording(span: Any) -> bool:
    return span is not None and (not hasattr(span, "is_recording") or span.is_recording())


def flush_traces(timeout_millis: int = 3000) -> None:
    """Сразу отправить закрытые spans в Phoenix, чтобы UI обновлялся во время долгой ячейки."""
    if not PHOENIX_FORCE_FLUSH or tracer_provider is None:
        return
    try:
        tracer_provider.force_flush(timeout_millis=timeout_millis)
    except TypeError:
        tracer_provider.force_flush()
    except Exception as exc:
        # Не ломаем исследовательский запуск из-за проблем наблюдаемости.
        print(f"Phoenix force_flush warning: {type(exc).__name__}: {str(exc)[:160]}")


class maybe_span:
    """Маленький no-op context manager, чтобы код работал и без Phoenix."""
    def __init__(self, name: str, *, kind: str = "CHAIN", input_value: Any = None, **attrs: Any):
        self.name = name
        self.kind = kind
        self.input_value = input_value
        self.attrs = attrs
        self._span_cm = None
        self._span = None

    def __enter__(self):
        if tracer is not None:
            self._span_cm = tracer.start_as_current_span(self.name)
            self._span = self._span_cm.__enter__()
            self._span.set_attribute("openinference.span.kind", self.kind)
            for key, value in self.attrs.items():
                self._span.set_attribute(key, str(value))
            if self.input_value is not None:
                self.set_input(self.input_value)
        return self

    def __exit__(self, exc_type, exc, tb):
        if self._span is not None and Status is not None and StatusCode is not None:
            if exc_type is None:
                self._span.set_status(Status(StatusCode.OK))
            else:
                self._span.record_exception(exc)
                self._span.set_status(Status(StatusCode.ERROR, str(exc)))
        if self._span_cm is not None:
            self._span_cm.__exit__(exc_type, exc, tb)
            flush_traces()

    def set_attribute(self, key: str, value: Any) -> None:
        if _span_is_recording(self._span):
            self._span.set_attribute(key, str(value))

    def set_input(self, value: Any) -> None:
        if _span_is_recording(self._span):
            self._span.set_attribute("input.value", _span_preview(value))
            self._span.set_attribute("input.mime_type", _span_mime_type(value))

    def set_output(self, value: Any) -> None:
        if _span_is_recording(self._span):
            self._span.set_attribute("output.value", _span_preview(value))
            self._span.set_attribute("output.mime_type", _span_mime_type(value))




## 3. Базовый слой модели: первый вызов модели и один вход для всех LLM-вызовов

Теперь подключим модель через OpenRouter. OpenRouter предоставляет OpenAI-совместимый API, поэтому мы используем обычный OpenAI SDK, но меняем `base_url` на `https://openrouter.ai/api/v1`.

В этой ячейке мы создаём не агента, а базовый слой для обращения к модели. Функция `call_model` станет единой точкой всех LLM-вызовов в блокноте: через неё позже будут проходить и обычные ответы, и вызовы модели с инструментами.

Сразу добавим несколько практических вещей: fallback-модель на случай ошибки, поддержку `tools` и `tool_choice`, а также span `model.call`, чтобы каждый вызов модели был виден в Phoenix.

Важно: пока модель только отвечает на переданные сообщения. Даже если функция уже умеет передавать инструменты, агентной системы ещё нет. Агент появится позже, когда вокруг модели возникнет цикл выполнения: состояние, план, обработка tool calls, вызовы инструментов, маршрутизация и правила остановки.


In [ ]:
def make_openrouter_client() -> Optional[Any]:
    if not LIVE_MODEL:
        return None
    return OpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url="https://openrouter.ai/api/v1",
        default_headers={
            "HTTP-Referer": "https://codefest.ru/",
            "X-Title": "CodeFest AI Researcher Masterclass",
        },
    )

client = make_openrouter_client()


def call_model(messages: list[dict[str, Any]], tools: Optional[list[dict[str, Any]]] = None, tool_choice: str = "auto") -> Any:
    """Единая точка live-вызова модели через OpenRouter."""
    if client is None:
        raise RuntimeError("OPENROUTER_API_KEY обязателен: вызовы модели выполняются через live API.")

    errors: list[str] = []
    for model in [PRIMARY_MODEL, FALLBACK_MODEL]:
        try:
            tool_names = [tool.get("function", {}).get("name") for tool in (tools or [])]
            with maybe_span(
                "model.call",
                kind="LLM",
                input_value={"model": model, "messages": messages, "tools": tool_names},
                model=model,
                tool_count=len(tools or []),
            ) as span:
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    tools=tools,
                    tool_choice=tool_choice if tools else None,
                    temperature=0.2,
                )
                message = response.choices[0].message
                span.set_output({
                    "content": message.content,
                    "tool_calls": [
                        {"name": call.function.name, "arguments": call.function.arguments}
                        for call in (getattr(message, "tool_calls", None) or [])
                    ],
                })
                return message
        except Exception as exc:
            errors.append(f"{model}: {type(exc).__name__}: {exc}")
    raise RuntimeError("Все model calls завершились ошибкой:\n" + "\n".join(errors))

if LIVE_MODEL:
    first_message = call_model([
        {"role": "system", "content": "Отвечай кратко и по-русски."},
        {"role": "user", "content": "В одном предложении объясни, зачем агенту нужна обвязка."},
    ])
    print(first_message.content)
else:
    print("Первый вызов модели пропущен: задайте OPENROUTER_API_KEY для live-вызовов OpenRouter.")



## 4. Инструменты: схема для модели и обработчик для кода

Tool calling не означает, что модель сама выполняет действие. Модель получает описание доступных инструментов и может вернуть структурированное намерение: какой инструмент вызвать и с какими JSON-аргументами. Само действие выполняет наш Python-код.

Поэтому у каждого инструмента в блокноте будут две стороны:

* **схема** — описание инструмента для модели в формате, совместимом с OpenAI API;
* **обработчик** — обычная Python-функция, которую вызовет наша обвязка.

В этой ячейке мы создадим типовое описание инструмента `ToolSpec`, реестр инструментов `ToolRegistry` и зарегистрируем первый простой инструмент `get_current_date`. Он возвращает текущую дату из переменной `TODAY`.

Пока мы только готовим инфраструктуру: описываем инструменты и складываем их в реестр. На следующем шаге модель сможет увидеть эти схемы, а позже мы добавим диспетчеризацию: разбор `tool_calls`, выбор нужного обработчика, выполнение функции и возврат результата модели сообщением `role="tool"`.


In [ ]:
@dataclass
class ToolSpec:
    name: str
    description: str
    parameters: dict[str, Any]
    handler: Callable[..., Any]

    def to_openai_tool(self) -> dict[str, Any]:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.parameters,
            },
        }

class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, ToolSpec] = {}

    def register(self, tool: ToolSpec) -> None:
        if tool.name in self._tools:
            raise ValueError(f"Инструмент уже зарегистрирован: {tool.name}")
        self._tools[tool.name] = tool

    def get(self, name: str) -> ToolSpec:
        if name not in self._tools:
            raise KeyError(f"Неизвестный инструмент: {name}")
        return self._tools[name]

    def schemas(self) -> list[dict[str, Any]]:
        return [tool.to_openai_tool() for tool in self._tools.values()]

    def names(self) -> list[str]:
        return list(self._tools)


def get_current_date() -> dict[str, str]:
    return {"today": TODAY}

current_date_tool = ToolSpec(
    name="get_current_date",
    description="Вернуть текущую дату в ISO-формате.",
    parameters={
        "type": "object",
        "properties": {},
        "additionalProperties": False,
    },
    handler=get_current_date,
)

registry = ToolRegistry()


registry.register(current_date_tool)

print("Зарегистрированные инструменты:", registry.names())
print(json.dumps(registry.schemas()[0], ensure_ascii=False, indent=2))




## 5. Цикл агента: модель вызывает инструмент

Теперь добавим минимальный цикл агента. До этого модель могла только ответить текстом или вернуть намерение вызвать инструмент. В этой ячейке наша обвязка впервые начнёт это намерение выполнять.

Логика цикла простая:

1. передаём модели историю сообщений и схемы доступных инструментов;
2. получаем ответ модели;
3. если модель вернула обычный текст — завершаем работу;
4. если модель вернула `tool_calls` — разбираем имя инструмента и JSON-аргументы;
5. находим нужный обработчик в `ToolRegistry`;
6. выполняем Python-функцию;
7. возвращаем результат модели сообщением `role="tool"`;
8. повторяем цикл, пока модель не даст финальный ответ или пока не сработает лимит итераций.

Это уже маленький ReAct-подобный loop: модель выбирает действие, код выполняет его, результат возвращается в контекст, и модель продолжает работу.

Важно: это ещё не весь harness исследовательского агента. Пока здесь нет Tavily, планировщика, состояния с `todos`, подагентов и полноценного управления контекстом. Но именно на этом шаге появляется ключевой механизм: модель перестаёт быть просто генератором текста и начинает работать через управляемый цикл «решение → действие → наблюдение».


In [ ]:
def _tool_call_name(call: Any) -> str:
    return call.function.name


def _tool_call_args(call: Any) -> dict[str, Any]:
    raw = call.function.arguments or "{}"
    if isinstance(raw, dict):
        return raw
    return json.loads(raw)


def _tool_call_id(call: Any) -> str:
    return getattr(call, "id", "mock-tool-call-id")


def dispatch_tool(call: Any, registry: ToolRegistry) -> dict[str, Any]:
    name = _tool_call_name(call)
    args = _tool_call_args(call)
    tool = registry.get(name)
    with maybe_span(
        "tool.dispatch",
        kind="TOOL",
        input_value={"tool_name": name, "args": args},
        tool_name=name,
        tool_args=args,
    ) as span:
        result = tool.handler(**args)
        span.set_output(result)
        return result


def assistant_message_to_dict(message: Any) -> dict[str, Any]:
    tool_calls = getattr(message, "tool_calls", None) or []
    payload: dict[str, Any] = {
        "role": "assistant",
        "content": message.content or "",
    }
    if tool_calls:
        payload["tool_calls"] = [
            {
                "id": _tool_call_id(call),
                "type": "function",
                "function": {
                    "name": _tool_call_name(call),
                    "arguments": json.dumps(_tool_call_args(call), ensure_ascii=False),
                },
            }
            for call in tool_calls
        ]
    return payload


def run_agent(
    user_prompt: str,
    registry: ToolRegistry,
    system_prompt: str,
    max_iterations: int = 4,
) -> str:
    messages: list[dict[str, Any]] = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    for iteration in range(1, max_iterations + 1):
        with maybe_span("agent.iteration", kind="AGENT", input_value={"iteration": iteration, "max_iterations": max_iterations}, iteration=iteration, max_iterations=max_iterations):
            message = call_model(messages, tools=registry.schemas())
            messages.append(message if isinstance(message, dict) else assistant_message_to_dict(message))
            tool_calls = getattr(message, "tool_calls", None) or []
            if not tool_calls:
                return message.content or ""

            for call in tool_calls:
                result = dispatch_tool(call, registry)
                messages.append({
                    "role": "tool",
                    "tool_call_id": _tool_call_id(call),
                    "name": _tool_call_name(call),
                    "content": json.dumps(result, ensure_ascii=False),
                })

    return "Достигнут лимит итераций. Агент остановлен обвязкой, а не моделью."

loop_demo = run_agent(
    "Какая сегодня дата? Используй инструмент, если он доступен.",
    registry,
    system_prompt="Ты учебный агент. Если нужен внешний факт, вызывай инструмент.",
    max_iterations=2,
)
print(loop_demo)




## 6. Состояние и задачи: находки и режим работы

У агента уже есть одно состояние — история сообщений. В ней хранится ход диалога: запрос пользователя, ответы модели, вызовы инструментов и результаты `role="tool"`. Это состояние нужно прежде всего самой модели: на каждом шаге мы передаём его в LLM, чтобы она могла продолжить работу.

Но для долгого исследовательского запуска одной истории сообщений мало. Обвязке тоже нужно собственное структурированное состояние: текущий режим работы, список задач, выполненные шаги, найденные факты, путь к итоговому отчёту и счётчик итераций.

В этой ячейке мы создадим `RunState` — состояние одного запуска агента. Оно не заменяет `messages`, а дополняет их. `messages` отвечают за контекст общения с моделью, а `RunState` помогает коду управлять процессом: отслеживать `todos`, сохранять находки, проверять готовность к финальному ответу и понимать, можно ли завершать работу.

Такой список задач не превращает учебного агента в промышленную систему, но уже даёт важный механизм контроля. Агент не просто продолжает диалог, а двигается по видимому состоянию, которое можно показать в трейсах, проверить в коде и использовать как условие остановки.


In [ ]:
Mode = Literal["plan", "execute", "final"]

@dataclass
class RunState:
    mode: Mode = "plan"
    todos: list[str] = field(default_factory=list)
    completed_todos: list[str] = field(default_factory=list)
    findings: list[dict[str, Any]] = field(default_factory=list)
    report_path: Optional[str] = None
    iteration_count: int = 0

    def add_todos(self, todos: list[str]) -> list[str]:
        added: list[str] = []
        existing = {todo.lower() for todo in self.todos + self.completed_todos}
        for todo in todos:
            clean = " ".join(todo.split())
            if clean and clean.lower() not in existing:
                self.todos.append(clean)
                existing.add(clean.lower())
                added.append(clean)
        return added

    def complete_todo(self, todo: str) -> bool:
        target = todo.strip().lower()
        for existing in list(self.todos):
            if existing.lower() == target:
                self.todos.remove(existing)
                self.completed_todos.append(existing)
                return True
        return False

    def remove_todos(self, todos: list[str]) -> tuple[list[str], list[str]]:
        removed: list[str] = []
        missing: list[str] = []
        for todo in todos:
            if self.complete_todo(todo):
                removed.append(todo)
            else:
                missing.append(todo)
        return removed, missing

    def is_incomplete(self) -> Optional[str]:
        if self.mode == "plan":
            return "Сначала вызови generate_plan и переведи задачу в execute mode."
        if self.todos:
            return "Остались незавершённые todos: " + "; ".join(self.todos)
        if self.report_path is None:
            return "Перед завершением вызови save_report и сохрани итоговый markdown-отчёт."
        return None

    def add_finding(self, question: str, notes: str, sources: list[str], evidence: Optional[list[dict[str, str]]] = None) -> None:
        self.findings.append({
            "question": question,
            "notes": notes,
            "sources": sources,
            "evidence": evidence or [],
        })

state = RunState()
state.add_todos(["Понять вопрос", "Собрать источники", "Сделать вывод"])
state.complete_todo("Понять вопрос")
print(state)



## 7. Планировщик как инструмент: из вопроса в ограниченный план

Теперь добавим планирование. В этой версии агент не будет возвращать план обычным текстом или JSON, который потом нужно парсить вручную. Планирование оформим как отдельный инструмент `generate_plan`.

Идея такая: пока агент находится в режиме `plan`, ему доступен только инструмент планирования. Модель должна вызвать `generate_plan(todos=[...])`, передать список исследовательских задач, а Python-код сохранит эти задачи в `RunState` и переведёт запуск в режим `execute`.

```text
model -> generate_plan(todos=[...]) -> state.mode = "execute"
```

Структуру плана задаёт схема инструмента: количество задач, тип данных и ограничение на лишние аргументы. Это надёжнее, чем просить модель «вернуть JSON» и потом разбирать текстовый ответ.

В этой ячейке мы подготовим сам инструмент планирования. Полный цикл, где главный агент сначала получает только `generate_plan`, а после планирования переключается на инструменты выполнения, появится следующим шагом.


In [ ]:
# DEMO_QUERY = (
#     "Сравни современные Python-фреймворки для построения AI-агентов: "
#     "LangGraph, LlamaIndex, CrewAI, AutoGen и Pydantic AI. "
#     "Найди их ключевые архитектурные идеи, сильные стороны, ограничения "
#     "и типичные сценарии применения. Сделай практическую рекомендацию: "
#     "какой фреймворк выбрать для учебного deep research agent и почему."
# )

DEMO_QUERY = (  
    "Проведи исследование и выбери лучший Python-фреймворк для создания учебного deep research agent. "
    "Сначала сам определи критерии выбора и найди релевантные open-source кандидаты. "
  "Затем составь shortlist из 4–6 фреймворков, сравни их по архитектуре, зрелости, "
  "наблюдаемости, поддержке итеративного поиска, human-in-the-loop, типизации и простоте объяснения участникам. "
  "В финале дай рекомендацию для мастер-класса: какой фреймворк выбрать, почему, и какие альтернативы стоит упомянуть."
)

def make_generate_plan_tool(run_state: RunState) -> ToolSpec:
    def generate_plan(todos: list[str]) -> dict[str, Any]:
        added = run_state.add_todos(todos[:6])
        run_state.mode = "execute"
        return {
            "result": "Plan accepted. Switch to execute mode and start using research tools.",
            "mode": run_state.mode,
            "todos": list(run_state.todos),
            "added": added,
        }

    return ToolSpec(
        name="generate_plan",
        description="Создать проверяемый план исследования и перевести агента из plan mode в execute mode.",
        parameters={
            "type": "object",
            "properties": {
                "todos": {
                    "type": "array",
                    "items": {"type": "string"},
                    "minItems": 3,
                    "maxItems": 6,
                    "description": "3–6 distinct исследовательских задач. Последняя задача должна проверять ссылки/цитирование.",
                },
            },
            "required": ["todos"],
            "additionalProperties": False,
        },
        handler=generate_plan,
    )


planner_demo_state = RunState(mode="plan")
planner_tool = make_generate_plan_tool(planner_demo_state)
print(json.dumps(planner_tool.to_openai_tool(), ensure_ascii=False, indent=2))




## 8. Инструмент Tavily: внешний источник фактов

Теперь добавим инструмент для веб-поиска. Для исследовательского агента это принципиальный шаг: свежие факты не должны браться «из памяти модели». Поиск в интернете — это внешнее действие, которое выполняет наша обвязка.

Модель будет видеть инструмент `search_web` и сможет запросить поиск по конкретной формулировке. Но сам запрос в Tavily выполнит Python-код, после чего результат вернётся модели как обычный результат инструмента.

Функция `search_web` возвращает список нормализованных результатов. Для каждого найденного источника мы сохраняем:

```python
{
    "title": ...,
    "url": ...,
    "content": ...,
    "raw_content": ...,
    "answer": ...,
    "score": ...,
    "source": "tavily",
}
```

Нормализация нужна, чтобы дальше агент работал не с сырым ответом Tavily, а с предсказуемой структурой: заголовок, ссылка, краткий фрагмент, markdown-контент, оценка релевантности и источник данных.

Если `TAVILY_API_KEY` отсутствует, Tavily недоступен или поиск вернул пустой результат, инструмент выбрасывает исключение. Это делает зависимость от внешнего поиска явной: ошибка будет видна в notebook и в трейсе Phoenix.


In [ ]:
def _trim_text(value: Any, limit: int = 1800) -> str:
    text = " ".join(str(value or "").split())
    if len(text) <= limit:
        return text
    return text[:limit].rstrip() + "…"


def normalize_tavily_results(raw: Any, source: str = "tavily") -> list[dict[str, str]]:
    if isinstance(raw, dict):
        items = raw.get("results", [])
        answer = raw.get("answer", "")
    else:
        items = raw or []
        answer = ""

    normalized: list[dict[str, str]] = []
    for item in items:
        normalized.append({
            "title": str(item.get("title", "Без названия")),
            "url": str(item.get("url", "")),
            "content": _trim_text(item.get("content") or item.get("snippet") or "", 900),
            "raw_content": _trim_text(item.get("raw_content") or item.get("rawContent") or "", 1800),
            "answer": _trim_text(answer, 900),
            "score": str(item.get("score", "")),
            "source": source,
        })
    return normalized


def require_live_search() -> None:
    if not LIVE_SEARCH:
        raise RuntimeError("TAVILY_API_KEY обязателен: поиск выполняется через Tavily API.")


def search_web(query: str, max_results: int = 8) -> list[dict[str, str]]:
    require_live_search()
    from tavily import TavilyClient

    with maybe_span(
        "tool.search_web",
        kind="TOOL",
        input_value={"query": query, "max_results": max_results},
        query=query,
        max_results=max_results,
        live=True,
    ) as span:
        tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
        raw = tavily.search(
            query=query,
            max_results=max_results,
            search_depth="advanced",
            include_answer="advanced",
            include_raw_content="markdown",
            include_usage=True,
            auto_parameters=True,
            timeout=45,
        )
        results = normalize_tavily_results(raw, source="tavily")
        if not results:
            raise RuntimeError(f"Tavily вернул пустой результат для запроса: {query}")
        span.set_output([
            {"title": item["title"], "url": item["url"], "score": item.get("score", "")}
            for item in results
        ])
        return results

search_tool = ToolSpec(
    name="search_web",
    description="Искать в интернете через Tavily и возвращать расширенный контекст: answer, raw markdown, content, score и URL.",
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Поисковый запрос. Язык выбирается по контексту задачи и ожидаемым источникам."},
            "max_results": {"type": "integer", "description": "Сколько результатов вернуть", "default": 8},
        },
        "required": ["query"],
        "additionalProperties": False,
    },
    handler=search_web,
)

registry.register(search_tool)
print("Инструмент поиска зарегистрирован:", search_tool.name)
if LIVE_SEARCH:
    print("Пример поиска:")
    for result in search_web("LangGraph official documentation stateful agents checkpointing", max_results=2):
        print(f"- {result['title']} — {result['url']} [{result['source']}] :: {result['content'][:120]}")
else:
    print("Пример Tavily пропущен: задайте TAVILY_API_KEY.")



## 9. Поисковый подагент: делегирование в чистый контекст

Теперь добавим поискового подагента. Его задача уже не просто выполнить один поиск, а провести маленькое самостоятельное исследование по делегированному вопросу.

Главный агент вызывает инструмент `delegate_search` и передаёт один или несколько исследовательских вопросов. Для каждого такого вопроса обвязка запускает отдельного поискового подагента. Подагент получает чистый контекст, собственный короткий agent loop и только один инструмент — `search_web`.

Подагент сам формулирует поисковые запросы и вызывает Tavily. Но полностью на усмотрение модели мы это не оставляем: код проверяет, что подагент сделал минимум два вызова `search_web`. Если модель пытается завершиться раньше, обвязка возвращает её обратно к поиску.

Такое делегирование помогает разгрузить главный контекст. Главному агенту не нужно держать все промежуточные поисковые шаги: он получает уже сжатые заметки, список источников и evidence по каждому делегированному вопросу.

В системном промпте написаны инструкции относительно языков: английский — для глобальных технических и официальных источников, русский — для русскоязычных тем, локальный язык — для локальных фактов. Итоговые заметки подагент всегда пишет на русском.


In [ ]:
def collect_urls_from_text(text: str) -> list[str]:
    urls = re.findall(r"https?://[^\s)\]>]+", text)
    return list(dict.fromkeys(urls))


def evidence_from_results(results: list[dict[str, str]], limit: int = 12) -> list[dict[str, str]]:
    evidence: list[dict[str, str]] = []
    seen: set[str] = set()
    for item in results:
        url = item.get("url", "")
        if not url or url in seen:
            continue
        seen.add(url)
        evidence.append({
            "title": item.get("title", ""),
            "url": url,
            "snippet": item.get("content") or item.get("answer") or item.get("raw_content", "")[:500],
        })
        if len(evidence) >= limit:
            break
    return evidence

SEARCH_SUBAGENT_SYSTEM_PROMPT = f"""
Ты поисковый подагент deep research системы. Сегодня {TODAY}.
У тебя есть только один инструмент: search_web, который делает Tavily search.

Твоя роль узкая: собрать проверяемые заметки по делегированному вопросу для главного агента.
Ты не главный агент: не пиши финальный отчёт, не выбирай победителя во всей задаче, не меняй критерии сравнения и не расширяй область исследования без необходимости.

Правила:
1. Не отвечай из памяти. Для каждого задания вызови search_web минимум два раза с разными запросами.
2. Язык запроса выбирай по контексту: английский для глобальных технических/официальных источников, русский для русскоязычных тем, локальный язык для локальных источников.
3. Отфильтровывай нерелевантные результаты. Если запрос ушёл в слишком общий web/frameworks-контекст вместо AI-agent frameworks, сделай уточняющий запрос.
4. Итоговые заметки всегда пиши на русском.
5. Пиши сжато, но содержательно: дай факты, ограничения, противоречия и ссылки, а не самостоятельный обзорный отчёт.
6. Каждое важное фактическое утверждение снабжай markdown-ссылкой.
7. Не используй заголовок первого уровня `#`. Верни структуру: "Краткий ответ", "Факты с источниками", "Ограничения/спорные места", "Источники".
8. В разделе "Источники" перечисли все использованные URL.
""".strip()


def run_search_subagent(question: str, max_iterations: int = 6) -> dict[str, Any]:
    if not LIVE_MODEL:
        raise RuntimeError("OPENROUTER_API_KEY обязателен: search subagent должен вызывать live model.")
    require_live_search()

    sub_registry = ToolRegistry()
    sub_registry.register(search_tool)
    messages: list[dict[str, Any]] = [
        {"role": "system", "content": SEARCH_SUBAGENT_SYSTEM_PROMPT},
        {"role": "user", "content": f"Исследовательский вопрос: {question}"},
    ]
    search_count = 0
    all_results: list[dict[str, str]] = []

    with maybe_span("subagent.search", kind="AGENT", input_value={"question": question}, question=question) as span:
        for iteration in range(1, max_iterations + 1):
            message = call_model(messages, tools=sub_registry.schemas())
            messages.append(assistant_message_to_dict(message))
            tool_calls = getattr(message, "tool_calls", None) or []

            if not tool_calls:
                if search_count >= 2:
                    notes = message.content or ""
                    sources = collect_urls_from_text(notes)
                    if not sources:
                        sources = [item["url"] for item in all_results if item.get("url")]
                    payload = {
                        "question": question,
                        "notes": notes,
                        "sources": list(dict.fromkeys(sources)),
                        "evidence": evidence_from_results(all_results),
                        "search_count": search_count,
                    }
                    span.set_output({
                        "question": question,
                        "search_count": search_count,
                        "source_count": len(payload["sources"]),
                        "notes_preview": notes[:1200],
                    })
                    return payload
                messages.append({
                    "role": "user",
                    "content": "Ты ещё не сделал минимум два Tavily search calls. Вызови search_web с уточняющим запросом.",
                })
                continue

            for call in tool_calls:
                result = dispatch_tool(call, sub_registry)
                if _tool_call_name(call) == "search_web":
                    search_count += 1
                    all_results.extend(result)
                messages.append({
                    "role": "tool",
                    "tool_call_id": _tool_call_id(call),
                    "name": _tool_call_name(call),
                    "content": json.dumps(result, ensure_ascii=False),
                })

    raise RuntimeError(f"Search subagent reached iteration limit before final notes: {question}")


def delegate_search(queries: list[str]) -> list[dict[str, Any]]:
    answers = []
    selected_queries = queries[:3]
    with maybe_span(
        "agent.delegate_search",
        kind="AGENT",
        input_value={"queries": selected_queries},
        query_count=len(selected_queries),
    ) as span:
        for query in selected_queries:
            answers.append(run_search_subagent(query))
        span.set_output([
            {"question": item["question"], "search_count": item.get("search_count"), "source_count": len(item.get("sources", []))}
            for item in answers
        ])
    return answers

print("Поисковый подагент готов: во время live-запуска он будет сам вызывать search_web.")



## 10. Итоговый отчёт: главный исследовательский агент

Теперь соберём полный исследовательский цикл. Главный агент больше не просто вызывает модель или отдельный инструмент: он управляет планом, делегирует поиск подагентам, обновляет задачи, синтезирует выводы и сохраняет итоговый markdown-отчёт.

Общий сценарий выглядит так:

```text
main model
  -> generate_plan(...)
  -> delegate_search([...])      # запускает поисковых подагентов
       subagent model
         -> search_web(...)
         -> search_web(...)
         -> заметки с источниками
  -> modify_todo(...)
  -> save_report(filename="report.md", markdown="...")
  -> краткий ответ пользователю
```

Главный агент не ищет в интернете напрямую. Для этого он использует `delegate_search`: каждый поисковый подагент получает отдельный вопрос, делает несколько Tavily-запросов и возвращает сжатые заметки, источники и evidence.

Python-обвязка исполняет вызовы инструментов, хранит состояние запуска в `RunState`, проверяет незавершённые `todos` и не даёт агенту завершиться, пока итоговый markdown-отчёт не сохранён через `save_report`.

Это всё ещё учебная версия: здесь нет песочницы, прав доступа, долговременной памяти, восстановления сессий и формального контура оценки качества. Но основные элементы долгого исследовательского агента уже собраны: планирование, инструменты, подагенты, состояние, ограничения и наблюдаемость.


<img src="схема ресерчера.png" width="900">

In [ ]:
MAIN_AGENT_SYSTEM_PROMPT = f"""
Ты учебный исследовательский агент. Сегодня {TODAY}.

Ты решаешь задачу через доступные инструменты: строишь план, делегируешь поиск подагентам, обновляешь задачи и сохраняешь итоговый markdown-отчёт.
Главный агент отвечает за план, критерии сравнения, синтез, финальные выводы и сохранение отчёта.
Поисковые подагенты — только поставщики проверяемых заметок и источников.

Режимы:
1. В начале обязательно вызови generate_plan с 3–6 проверяемыми todos.
2. После generate_plan исследуй задачи через delegate_search. Это единственный способ веб-поиска для главного агента.
3. delegate_search принимает 1–3 distinct вопроса для поисковых подагентов. Формулируй их как узкие исследовательские вопросы: один аспект, один фреймворк или один тип доказательств на подагента.
4. Не делегируй подагенту роль главного автора: не проси его написать весь отчёт, выбрать общего победителя или сравнить всё сразу, если это должен синтезировать ты.
5. После получения результатов используй modify_todo, чтобы закрывать выполненные пункты или добавлять важные rabbit holes.
6. Финальный отчёт пиши на русском. Поисковые вопросы и запросы подагентов могут быть на языке лучших источников.

Правила работы с источниками:
7. Различай уровень источника:
   - официальная документация, changelog, GitHub организации/проекта и публикации команды продукта — основной источник фактов о возможностях, API, архитектуре и статусе проекта;
   - issue/PR/discussion в официальном репозитории — сильный сигнал о реальных ограничениях, но формулируй аккуратно;
   - независимые блоги, обзоры, benchmark-посты, Medium/dev.to/Substack — мнения, интерпретации и практический опыт, а не окончательная истина;
   - SEO-сравнения и vendor-блоги — полезны для ориентира, но не должны быть единственной опорой для сильного вывода.
8. Не выдавай мнение блогера за факт. Пиши "по оценке автора обзора", "в сравнительном обзоре утверждается", "официальная документация подтверждает".
9. Для спорных утверждений указывай уровень уверенности или контекст: "вероятно", "по доступным источникам", "для новых проектов выглядит рискованно", а не абсолютные формулировки.
10. Если источники расходятся, покажи расхождение и объясни, какой источник сильнее и почему.

Правила финального отчёта:
11. Используй обычные markdown-ссылки только в формате `[Источник](https://...)`. Не используй двойные скобки вида `[[Источник]](https://...)`.
12. В отчёте должны быть inline markdown-ссылки или явная привязка фактов к URL, а в конце — раздел "Источники".
13. Рекомендацию формулируй как архитектурный выбор под заданные критерии, а не как абсолютного победителя рынка. Если уместен гибрид, явно объясни роли компонентов.
14. Отделяй факты из официальных источников от выводов на основе обзоров и собственного синтеза.
15. Перед финальным ответом обязательно вызови save_report и сохрани markdown в файл report.md.
16. В чат не выводи весь отчёт; после save_report дай короткое резюме и путь к файлу.
""".strip()


def make_modify_todo_tool(run_state: RunState) -> ToolSpec:
    def modify_todo(action: str, todos: list[str]) -> dict[str, Any]:
        if action == "add":
            added = run_state.add_todos(todos)
            return {"action": action, "added": added, "todos": list(run_state.todos)}
        if action == "remove":
            removed, missing = run_state.remove_todos(todos)
            return {"action": action, "removed": removed, "missing": missing, "todos": list(run_state.todos)}
        raise ValueError("action must be 'add' or 'remove'")

    return ToolSpec(
        name="modify_todo",
        description="Добавить новые todos или отметить существующие todos выполненными.",
        parameters={
            "type": "object",
            "properties": {
                "action": {"type": "string", "enum": ["add", "remove"]},
                "todos": {"type": "array", "items": {"type": "string"}, "minItems": 1},
            },
            "required": ["action", "todos"],
            "additionalProperties": False,
        },
        handler=modify_todo,
    )


def make_delegate_search_tool(run_state: RunState) -> ToolSpec:
    def delegate_search_tool_handler(queries: list[str]) -> dict[str, Any]:
        results = delegate_search(queries[:3])
        for item in results:
            run_state.add_finding(item["question"], item["notes"], item["sources"], item["evidence"])
        return {"results": results, "finding_count": len(run_state.findings)}

    return ToolSpec(
        name="delegate_search",
        description="Делегировать 1–3 distinct исследовательских вопроса search subagents. Каждый subagent сам вызывает Tavily search_web.",
        parameters={
            "type": "object",
            "properties": {
                "queries": {
                    "type": "array",
                    "items": {"type": "string"},
                    "minItems": 1,
                    "maxItems": 3,
                    "description": "Вопросы для подагентов. Формулируй как исследовательские вопросы, не как однословные keywords.",
                },
            },
            "required": ["queries"],
            "additionalProperties": False,
        },
        handler=delegate_search_tool_handler,
    )


def make_save_report_tool(run_state: RunState) -> ToolSpec:
    def save_report(filename: str, markdown: str) -> dict[str, Any]:
        path = Path(filename or "report.md")
        if path.is_absolute() or ".." in path.parts:
            raise ValueError("filename must be a relative path inside the notebook working directory")
        if path.suffix.lower() != ".md":
            path = path.with_suffix(".md")
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(markdown, encoding="utf-8")
        run_state.report_path = str(path)
        run_state.mode = "final"
        return {"result": "report_saved", "path": str(path), "chars": len(markdown)}

    return ToolSpec(
        name="save_report",
        description="Сохранить итоговый исследовательский отчёт в markdown-файл.",
        parameters={
            "type": "object",
            "properties": {
                "filename": {"type": "string", "description": "Относительный путь к .md файлу, например report.md"},
                "markdown": {"type": "string", "description": "Полный markdown отчёт на русском с источниками"},
            },
            "required": ["filename", "markdown"],
            "additionalProperties": False,
        },
        handler=save_report,
    )


def make_research_registry(run_state: RunState) -> ToolRegistry:
    active_registry = ToolRegistry()
    active_registry.register(make_generate_plan_tool(run_state))
    active_registry.register(make_modify_todo_tool(run_state))
    active_registry.register(make_delegate_search_tool(run_state))
    active_registry.register(make_save_report_tool(run_state))
    return active_registry


def require_live_agent_runtime() -> None:
    if not LIVE_MODEL:
        raise RuntimeError("OPENROUTER_API_KEY обязателен: главный агент должен вызывать live model.")
    require_live_search()


def run_researcher(query: str, max_iterations: int = 24) -> dict[str, Any]:
    require_live_agent_runtime()
    run_state = RunState(mode="plan")
    research_registry = make_research_registry(run_state)
    messages: list[dict[str, Any]] = [
        {"role": "system", "content": MAIN_AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": query},
    ]

    with maybe_span("agent.full_researcher", kind="AGENT", input_value={"query": query}, query=query, max_iterations=max_iterations) as run_span:
        for iteration in range(1, max_iterations + 1):
            run_state.iteration_count = iteration
            with maybe_span(
                "agent.iteration",
                kind="AGENT",
                input_value={"iteration": iteration, "mode": run_state.mode, "pending_todos": list(run_state.todos)},
                iteration=iteration,
                mode=run_state.mode,
                pending_todos=len(run_state.todos),
            ) as iteration_span:
                message = call_model(messages, tools=research_registry.schemas())
                messages.append(assistant_message_to_dict(message))
                tool_calls = getattr(message, "tool_calls", None) or []

                if not tool_calls:
                    reason = run_state.is_incomplete()
                    if reason:
                        messages.append({"role": "user", "content": reason})
                        continue
                    payload = {
                        "final_message": message.content or "Отчёт сохранён.",
                        "report_path": run_state.report_path,
                        "state": run_state,
                    }
                    iteration_span.set_output({"final_message": payload["final_message"], "report_path": run_state.report_path})
                    run_span.set_output({"final_message": payload["final_message"], "report_path": run_state.report_path})
                    return payload

                iteration_span.set_output({"tool_calls": [_tool_call_name(call) for call in tool_calls]})
                for call in tool_calls:
                    result = dispatch_tool(call, research_registry)
                    messages.append({
                        "role": "tool",
                        "tool_call_id": _tool_call_id(call),
                        "name": _tool_call_name(call),
                        "content": json.dumps(result, ensure_ascii=False),
                    })

        raise RuntimeError("Главный агент достиг лимита итераций. Остаток: " + (run_state.is_incomplete() or "нет деталей"))


if LIVE_MODEL and LIVE_SEARCH:
    researcher_result = run_researcher(DEMO_QUERY)
    final_report_path = researcher_result["report_path"]
    print(researcher_result["final_message"])
    print("Отчёт сохранён:", final_report_path)
else:
    print("Финальный исследовательский запуск пропущен: задайте OPENROUTER_API_KEY и TAVILY_API_KEY.")




## 11. Разбор запуска: что агент реально сделал

После финального ответа важно смотреть не только на `report.md`, но и на поведение всей системы. Исследовательский агент мог сделать разное количество итераций, запустить несколько подагентов, сформулировать разные поисковые запросы, потратить разное число токенов и по-разному пройти путь от плана к отчёту.

В этой ячейке мы собираем краткую сводку запуска. Сначала читаем сохранённый markdown-отчёт и считаем его размер, количество строк, markdown-ссылок и уникальных URL. Затем, если доступна локальная Phoenix SQLite DB, находим последний trace `agent.full_researcher` и анализируем его spans.

Из трейса можно увидеть:

* сколько было итераций главного агента;
* сколько раз вызывалась модель;
* какие инструменты реально выполнялись;
* сколько поисковых подагентов стартовало;
* какие Tavily-запросы были отправлены;
* сколько prompt/completion-токенов было потрачено;
* какие шаги заняли больше всего времени.

Если Phoenix недоступен или trace не найден, ячейка всё равно покажет метрики сохранённого отчёта. Это полезно: даже без UI мы можем получить минимальный post-run analysis прямо в notebook.


In [ ]:
import sqlite3
from collections import Counter
from datetime import datetime


def _parse_dt(value: str) -> datetime:
    return datetime.fromisoformat(value.replace(" ", "T"))


def _span_duration_seconds(row: Any) -> float:
    if not row["end_time"]:
        return 0.0
    return (_parse_dt(row["end_time"]) - _parse_dt(row["start_time"])).total_seconds()


def _read_span_attrs(row: Any) -> dict[str, Any]:
    try:
        return json.loads(row["attributes"] or "{}")
    except Exception:
        return {}


def _candidate_phoenix_dbs() -> list[Path]:
    candidates: list[Path] = []
    if PHOENIX_SESSION is not None:
        working_dir = getattr(PHOENIX_SESSION, "working_dir", None)
        if working_dir:
            candidates.append(Path(working_dir) / "phoenix.db")
    if os.getenv("PHOENIX_WORKING_DIR"):
        candidates.append(Path(os.environ["PHOENIX_WORKING_DIR"]) / "phoenix.db")
    candidates.extend([
        Path.cwd() / ".phoenix" / "phoenix.db",
        Path.cwd() / ".phoenix13" / "phoenix.db",
        Path.cwd().parent / ".phoenix" / "phoenix.db",
        Path.cwd().parent / ".phoenix13" / "phoenix.db",
    ])
    unique: list[Path] = []
    seen: set[Path] = set()
    for path in candidates:
        resolved = path.expanduser().resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique.append(resolved)
    return unique


def find_latest_research_trace() -> Optional[dict[str, Any]]:
    for db_path in _candidate_phoenix_dbs():
        if not db_path.exists():
            continue
        try:
            connection = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
            connection.row_factory = sqlite3.Row
            trace = connection.execute(
                """
                SELECT traces.id, traces.trace_id, traces.start_time, traces.end_time
                FROM traces
                JOIN spans ON spans.trace_rowid = traces.id
                WHERE spans.name = 'agent.full_researcher'
                ORDER BY traces.id DESC
                LIMIT 1
                """
            ).fetchone()
            if trace is None:
                connection.close()
                continue
            spans = connection.execute(
                """
                SELECT id, name, start_time, end_time, attributes, status_code,
                       llm_token_count_prompt, llm_token_count_completion
                FROM spans
                WHERE trace_rowid = ?
                ORDER BY id
                """,
                (trace["id"],),
            ).fetchall()
            connection.close()
            return {"db_path": db_path, "trace": dict(trace), "spans": spans}
        except Exception as exc:
            print(f"Не удалось прочитать Phoenix DB {db_path}: {type(exc).__name__}: {exc}")
    return None


def summarize_report(path: Optional[str]) -> dict[str, Any]:
    if not path:
        return {"path": None, "exists": False}
    report_path = Path(path)
    if not report_path.is_absolute():
        report_path = Path.cwd() / report_path
    if not report_path.exists():
        return {"path": str(report_path), "exists": False}
    text = report_path.read_text(encoding="utf-8")
    urls = re.findall(r"https?://[^\s)\]>]+", text)
    return {
        "path": str(report_path),
        "exists": True,
        "chars": len(text),
        "lines": text.count("\n") + 1,
        "markdown_links": len(re.findall(r"\[[^\]]+\]\(https?://", text)),
        "urls": len(urls),
        "unique_urls": len(set(urls)),
    }


def summarize_latest_run() -> dict[str, Any]:
    report_path = globals().get("final_report_path") or (globals().get("researcher_result") or {}).get("report_path")
    summary: dict[str, Any] = {"report": summarize_report(report_path)}

    trace_data = find_latest_research_trace()
    if trace_data is None:
        summary["phoenix"] = {"available": False, "message": "agent.full_researcher trace not found"}
        return summary

    trace = trace_data["trace"]
    spans = trace_data["spans"]
    names = Counter(row["name"] for row in spans)
    search_queries = []
    tool_dispatches = []
    for row in spans:
        attrs = _read_span_attrs(row)
        if row["name"] == "tool.search_web":
            search_queries.append(attrs.get("query"))
        if row["name"] == "tool.dispatch":
            tool_dispatches.append(attrs.get("tool_name"))

    prompt_tokens = sum(row["llm_token_count_prompt"] or 0 for row in spans)
    completion_tokens = sum(row["llm_token_count_completion"] or 0 for row in spans)
    started = _parse_dt(trace["start_time"])
    ended = _parse_dt(trace["end_time"])

    summary["phoenix"] = {
        "available": True,
        "db_path": str(trace_data["db_path"]),
        "ui": PHOENIX_UI_URL,
        "trace_id": trace["trace_id"],
        "duration_seconds": round((ended - started).total_seconds(), 1),
        "span_count": len(spans),
        "span_counts": dict(names),
        "tool_dispatch_counts": dict(Counter(tool_dispatches)),
        "search_queries": [query for query in search_queries if query],
        "token_usage": {
            "prompt": prompt_tokens,
            "completion": completion_tokens,
            "total": prompt_tokens + completion_tokens,
        },
        "longest_spans": [
            {
                "name": row["name"],
                "seconds": round(_span_duration_seconds(row), 1),
                "status": row["status_code"],
            }
            for row in sorted(spans, key=_span_duration_seconds, reverse=True)[:8]
        ],
    }
    return summary


def _format_seconds(seconds: Optional[float]) -> str:
    if seconds is None:
        return "n/a"
    minutes, sec = divmod(int(round(seconds)), 60)
    if minutes >= 60:
        hours, minutes = divmod(minutes, 60)
        return f"{hours}h {minutes}m {sec}s"
    if minutes:
        return f"{minutes}m {sec}s"
    return f"{sec}s"


def _format_int(value: Any) -> str:
    if value is None:
        return "0"
    return f"{int(value):,}".replace(",", " ")


def _format_status(status: Any) -> str:
    if status == "UNSET" or status is None:
        return "без ошибки"
    if status == "ERROR":
        return "ошибка"
    if status == "OK":
        return "ok"
    return str(status)


def render_run_summary_markdown(summary: dict[str, Any]) -> str:
    report = summary.get("report", {})
    phoenix = summary.get("phoenix", {})
    lines: list[str] = ["# Что сделал агент", ""]

    lines.extend(["## Отчёт"])
    if report.get("exists"):
        lines.extend([
            f"- **Файл:** `{report['path']}`",
            f"- **Размер:** {_format_int(report.get('chars'))} символов, {_format_int(report.get('lines'))} строк",
            f"- **Ссылки:** {_format_int(report.get('markdown_links'))} markdown-ссылок, {_format_int(report.get('unique_urls'))} уникальных URL",
        ])
    else:
        lines.append("- Отчёт не найден. Сначала запустите ячейку **Final report**.")

    lines.extend(["", "## Phoenix trace"])
    if not phoenix.get("available"):
        lines.append(f"- Trace не найден: {phoenix.get('message', 'Phoenix DB недоступна')}")
        return "\n".join(lines)

    lines.extend([
        f"- **UI:** {phoenix.get('ui') or 'n/a'}",
        f"- **Trace ID:** `{phoenix.get('trace_id')}`",
        f"- **Длительность:** {_format_seconds(phoenix.get('duration_seconds'))}",
        f"- **Spans:** {_format_int(phoenix.get('span_count'))}",
    ])

    tokens = phoenix.get("token_usage", {})
    lines.extend([
        "",
        "## Стоимость контекста",
        "| Prompt tokens | Completion tokens | Total tokens |",
        "|---:|---:|---:|",
        f"| {_format_int(tokens.get('prompt'))} | {_format_int(tokens.get('completion'))} | {_format_int(tokens.get('total'))} |",
    ])

    dispatch = phoenix.get("tool_dispatch_counts", {})
    span_counts = phoenix.get("span_counts", {})
    action_rows = [
        ("Итерации главного агента", span_counts.get("agent.iteration", 0)),
        ("Вызовы модели", span_counts.get("model.call", 0)),
        ("Tool dispatch всего", span_counts.get("tool.dispatch", 0)),
        ("Планы generate_plan", dispatch.get("generate_plan", 0)),
        ("Делегирования delegate_search", dispatch.get("delegate_search", 0)),
        ("Search subagents", span_counts.get("subagent.search", 0)),
        ("Tavily search_web", dispatch.get("search_web", span_counts.get("tool.search_web", 0))),
        ("Изменения todo", dispatch.get("modify_todo", 0)),
        ("Сохранения отчёта", dispatch.get("save_report", 0)),
    ]
    lines.extend(["", "## Агентные действия", "| Что считаем | Количество |", "|---|---:|"])
    lines.extend(f"| {label} | {_format_int(value)} |" for label, value in action_rows)

    longest = phoenix.get("longest_spans", [])
    if longest:
        lines.extend(["", "## Самые долгие шаги", "| Span | Время | Статус |", "|---|---:|---|"])
        for item in longest[:6]:
            lines.append(f"| `{item.get('name')}` | {_format_seconds(item.get('seconds'))} | {_format_status(item.get('status'))} |")

    queries = phoenix.get("search_queries", [])
    if queries:
        lines.extend(["", "## Реальные Tavily-запросы"])
        lines.extend(f"{idx}. {query}" for idx, query in enumerate(queries, start=1))

    lines.extend(["", "> Для дополнительного разбора сырые данные остаются в переменной `run_summary`."])
    return "\n".join(lines)


def show_run_summary(summary: dict[str, Any]) -> None:
    markdown_text = render_run_summary_markdown(summary)
    try:
        from IPython.display import Markdown, display

        display(Markdown(markdown_text))
    except Exception:
        print(markdown_text)


run_summary = summarize_latest_run()
show_run_summary(run_summary)


## 12. Куда развивать систему: соревновательная часть и улучшения baseline

Мы собрали учебного исследовательского агента. Он уже умеет строить план, вызывать инструменты, делегировать поиск подагентам, сохранять отчёт и оставлять трейсы в Phoenix. Но это базовая версия, которую можно улучшать.

Дальше ваша задача — выбрать одно или несколько направлений доработки и проверить, как они влияют на качество итогового исследования.

Возможные направления:

* **Улучшить планирование**: сделать разбиение задачи точнее, добавить критерии хорошего плана, убрать слишком общие или пересекающиеся задачи.
* **Улучшить поисковые запросы**: добавлять уточняющие формулировки, отдельно искать официальную документацию, GitHub-репозитории, обсуждения, журналы изменений и независимые обзоры.
* **Повысить качество источников**: отделять официальные источники от блогов, рекламных сравнений и поверхностных обзоров; меньше опираться на слабые источники.
* **Навести порядок в цитировании**: требовать, чтобы важные утверждения были связаны с конкретными ссылками, а не просто сопровождались списком URL в конце.
* **Добавить таблицу доказательств**: для ключевых выводов сохранять связку «утверждение → источник → насколько надёжно подтверждено».
* **Показывать степень уверенности**: помечать выводы как «подтверждено официально», «есть в независимых обзорах», «следует из косвенных признаков», «требует дополнительной проверки».
* **Улучшить промпт поискового подагента**: научить его лучше отличать официальные источники, делать уточняющие запросы и не приносить слабые результаты.
* **Добавить проверку пробелов перед `save_report`**: перед сохранением отчёта проверять, все ли критерии покрыты, есть ли источники по каждому важному выводу и не осталось ли очевидных недоработок.
* **Добавить критика перед финальным отчётом**: отдельным шагом искать слабые места, неподтверждённые утверждения, плохие источники и логические скачки.
* **Зафиксировать структуру отчёта**: задать обязательные разделы, формат сравнения, таблицу критериев, блок ограничений и финальную рекомендацию.
* **Провести проверки с отключением частей системы**: например, убрать слабые источники, упростить планирование или ослабить промпт и посмотреть, как изменится результат.




После мастер-класса систему можно расширять отдельным слоем: **контур оценки качества** проверяет отчёты на наборах задач и помогает сравнивать изменения агента с baseline.


Из готовых фреймворков для дальнейшего развития можно сравнить **LangChain/LangGraph**, LlamaIndex, CrewAI, AutoGen и Pydantic AI.
